# FireSense: training on Google Colab
**Runtime > Change runtime type > T4 GPU** first.
Before running: put `firesense_project.zip` and the dataset zip (e.g. `D-Fire.zip`) in a Google Drive folder called `FireSense`.
Optional: a folder `FireSense/hard_negatives/` with images that contain NO fire/smoke (sunsets, fog, steam, red clothes, car lights).

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- paths: edit if your Drive folder is named differently ----
DRIVE = '/content/drive/MyDrive/FireSense'
PROJECT_ZIP = f'{DRIVE}/firesense_project.zip'
DATASET_ZIP = f'{DRIVE}/D-Fire.zip'
HARD_NEG    = f'{DRIVE}/hard_negatives'      # optional folder
RUNS        = f'{DRIVE}/runs'                # training outputs are saved to Drive (survive disconnects)

In [ ]:
!unzip -q -o "$PROJECT_ZIP" -d /content
%cd /content/firesense_project
!pip install -q -r requirements.txt
!mkdir -p /content/dfire && unzip -q -o "$DATASET_ZIP" -d /content/dfire
!ls /content/dfire

## Stages 2-3: audit, clean, de-duplicate, split
If the unzip created an extra folder level, point `--src` at the folder that directly contains `train/` and `test/`.

In [ ]:
import os
SRC = '/content/dfire'
HN = f'--hard-negatives {HARD_NEG}' if os.path.isdir(HARD_NEG) else ''
!python scripts/prepare_dataset.py --src $SRC --out /content/data/firesense $HN

## Stage 6: optional synthetic-haze augmentation (more smoke robustness)

In [ ]:
!python scripts/augment_haze.py --data /content/data/firesense --ratio 0.3

## Stage 7: train YOLO
**Fast profile** (first run, roughly an hour on a T4; GPU speed varies): `yolov8n.pt`, `--imgsz 512`, `--epochs 25`, `--fraction 0.5`.
**Full profile**: `yolov8s.pt`, `--imgsz 640`, `--epochs 40`, `--fraction 1.0` (several hours).

In [ ]:
!python scripts/train.py --data /content/data/firesense/data.yaml \
    --model yolov8n.pt --imgsz 512 --epochs 25 --batch 32 --fraction 0.5 \
    --project "$RUNS" --name firesense_fast

## Stage 9: evaluation (mAP, confusion matrix, PR curves, ablation table, failure cases)

In [ ]:
!python scripts/evaluate.py --weights models/best.pt --data /content/data/firesense/data.yaml --max-images 1500

## Stage 10: explainability heatmaps

In [ ]:
!python scripts/explain.py --images /content/data/firesense/test/images --weights models/best.pt --n 12

## Show a result figure + save everything to Drive

In [ ]:
from IPython.display import Image, display
display(Image('results/ablation.png'))
!mkdir -p "$DRIVE/outputs"
!cp models/best.pt "$DRIVE/outputs/best.pt"
!cp -r results "$DRIVE/outputs/results"
print('Saved best.pt and results to', DRIVE + '/outputs')

Download `outputs/best.pt` from Drive, put it in your local project's `models/` folder, and run `streamlit run app.py`.